<div align="left">

# 第 1 册：初识 CANNBot——让 AI 帮你写 NPU 算子

你多半用过 ChatGPT、DeepSeek 这类 AI 聊天助手。但如果你是开发者，你真正想要的可能不是「AI 告诉我怎么做」，而是**AI 直接帮我把活干了**——帮我把算子写了、测试跑了、报错修了。

这正是 **CANNBot** 存在的意义：一支专门为昇腾 NPU 开发而组建的 AI 团队。本册用约 10 分钟回答三个问题：

1. **编程 Agent 和聊天 AI 有什么区别？**（为什么说一个在「出主意」，一个在「干活」）
2. **CANNBot 是什么、怎么组织的？**（三层架构与工作流）
3. **它背后是一个怎样的生态？**（六仓全景一图流）

本册不涉及任何安装与实操——下一册你会亲手让它交付第一个算子。

> **前置要求**：具备 NPU/CANN 基础认知（学过[快速上手](https://gitcode.com/cann/cann-learning-hub/tree/master/quick_start)或 [Ascend C 教程（Kernel 直调）第 1 章](./../../ascendc_operator_development_light/01_basic_overview/01.03_cann_arch_ascend_npu_principle.ipynb)水平即可）。

</div>

<div align="left">

## 1.1 聊天 AI vs 编程 Agent：出主意 vs 干活

假设你想给昇腾 NPU 写一个 `add`（矢量加法）算子，分别找两者帮忙：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">对比维度</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">聊天 AI（顾问）</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编程 Agent（员工）</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">能看什么</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">只看到你贴进对话框的内容</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">主动读你的工程目录、配置、已有代码</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">能做什么</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">输出文字和代码片段，靠你复制粘贴</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">创建/修改文件、执行命令、编译、跑测试</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">怎么确认结果</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">「看起来是对的」，对错你自己验</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">自己运行测试、比对结果，用通过/失败说话</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">交付物</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一段聊天记录</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一整套可编译工程：代码、测试、文档</td></tr>
</tbody></table>

本质区别是**闭环**：

```text
理解需求 → 写代码 → 执行（编译/运行）→ 看结果 → 改代码 → 再执行 → …… → 验证通过，交付
```

聊天 AI 的流程到「给出代码」就停了，后半段全靠人；编程 Agent 把整个闭环握在自己手里。

### 三个关键词

- **Agent（智能体）**＝大模型（大脑）＋工具（手脚：读写文件、执行命令）＋自主决策（你给目标，它定步骤）。承载 Agent 运行的软件叫**宿主**，如 OpenCode、Claude Code、Cursor；
- **Skill（技能）**＝写给 Agent 看的领域知识手册。Agent 不可能天生懂「Ascend C 的 Tiling 怎么切」，把正确做法写成手册、干活前按需加载，出错率大幅下降。Skill 人人可写、越积越厚——这是社区共建的核心；
- **多 Agent 协作**＝一支团队而不是一个人。PM 只调度不干活，架构师设计、开发者编码、测试者验收——**写代码的不能自己宣布通过**，和人类工程的「开发/测试分离」同理。

> ⚠️ **一个重要提醒**：Agent 生成的代码受模型、语料、指令影响，**不能保证完全正确**——必须独立验证后才能使用。这不是缺陷告白，而是 CANNBot 把「独立验收」设计进工作流的根本原因。

</div>

<div align="left">

## 1.2 CANNBot 是什么

一句话定位：

> **CANNBot 是面向 CANN 开发的系列智能体，把昇腾开发中「知识密集 + 流程标准」的环节交给 AI 团队自动完成，让你聚焦决策本身。**

它能覆盖的场景以算子开发为主线，辐射四大块：**Ascend C 算子开发**（Kernel 直调/注册算子——「帮我开发一个 add 算子，float32」）、**多语言路径**（PyPTO/TileLang/Triton/Catlass）、**推理与性能**（模型推理优化、算子性能演进）、**迁移与配套**（CUDA→Ascend 迁移、质量保障）。目前全生态已沉淀 **100+ 款 Skill**。

同样重要的是边界——它**不保证代码完全正确**（所以必须独立验收）、**不替你理解需求**（需求越含糊它发挥越自由）、**不是黑盒**（每步留痕，你始终是管理者）。

### 三层架构：菜谱、厨师与手册

> **Plugin 像一道菜的完整菜谱，Agent 是按岗位分工的厨师，Skill 是岗位手边的知识手册。**

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">层次</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">回答的问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">例子</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Plugin（应用编排层）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">按什么流程干</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">ops-direct-invoke-flash（本课程实操用）</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Agent（角色执行层）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">谁来干这一步</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">architect / developer / reviewer</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Skill（知识能力层）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">怎么干才对</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">ascendc-api-best-practices、tiling-design</td></tr>
</tbody></table>

这个分层的关键好处：**新增一个场景＝新增一份菜谱（Plugin），厨师和手册直接复用**——这是几十个场景能快速铺开的结构原因。

<img src="./images/01_three_layer_architecture.png" alt="CANNBot 三层架构示意图" width="800px">

### 工作流：轻量版与完整版

本课程实操用的 **Flash 轻量版**（ops-direct-invoke-flash）走八个阶段的 TDD 主线：**环境探测与骨架 → 定义/设计文档 → 测试套件 → 核函数实现 → 真机验证 → 收尾**。需要更强过程管控的场景（正式交付、多人协作）可用完整版（ops-direct-invoke）：八阶段（0~7：开发准备 → 需求分析 → 方案设计 → 代码开发 → 性能验收 → 代码检视 → 上库准备 → 开发总结）＋六道**检查点门禁（CP0~CP5）**——验收方对执行方的产物独立把关，**不通过就打回**。两版共享同一套「AI 产出必须被独立验证」的原则。（完整版的阶段设计与 CP 门禁将在[进阶篇第 4 册](./04_complete_workflow.ipynb)深入拆解。）

</div>

<div align="left">

## 1.3 用代码看清：一次开发的完整链路

运行下面的代码，把「你说需求 → Agent 团队分工 → 交付验证过的算子」的链路画出来（本代码随处可跑，无需 NPU）：

</div>

In [ ]:
def draw_cannbot_pipeline():
    """画出一条需求在 CANNBot（Flash 版）中的完整流转。"""
    bar = "=" * 60
    print(f"{bar}")
    print("你说：「开发一个 add_custom 算子：z = x + y，float32，(8, 2048)」")
    print(f"{bar}")
    stages = [
        ("环境探测", "识别芯片（如 Ascend910B3），判定复杂度档位"),
        ("定义文档", "数学语义 / dtype 策略 / 边界场景"),
        ("设计文档", "tiling 方案 / UB 预算 / 校验记录"),
        ("测试先行", "参数化用例矩阵 + 边界值，先写好后验证"),
        ("内核实现", "CopyIn → 计算 → CopyOut 三级流水 + tiling"),
        ("真机验证", "编译 → pytest 在真实 NPU 上与 CPU 参考逐值比对"),
        ("收尾归档", "全绿状态 + 交付报告"),
    ]
    for name, desc in stages:
        print(f"  [{name}]  {desc}")
    print(f"{bar}")
    print("交付：operators/add_custom/{add_custom.asc, test_add_custom.py, CMakeLists.txt, docs/}")
    print()
    print("要点：文档先行、测试先行、真机验证——AI 产出必须被独立验证。")
    print("      复杂算子还会追加多 Agent 评审与测试门禁（图为 simple 档快速路径）。")

draw_cannbot_pipeline()

<div align="left">

## 1.4 全景一图：CANNBot 背后的六仓生态

CANNBot 不只是一个仓库，而是一组分工明确的仓库群（你在课程里直接接触的是前两个）:

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">仓库</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一句话定位</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot（应用仓）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">工作流编排 × 场景 Plugin，按场景组织的「应用商店」</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-skills</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">知识手册总库（100+ 款 Skill，6 大域分区）——下册你要克隆的就是它</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-knowledge</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">更新快的「活知识」（API 语义/错误码/芯片规格）放库里用时再查</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-dsl</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">为 Agent 重设计的算子语言：易错的硬件表达交给编译器</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cann-bench</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子能力公共标尺（评测 + 排行榜）——第 3 册详述</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-sentry</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">观测 × 审计：让 Agent 干活全过程「看得见、查得信」</td></tr>
</tbody></table>

现阶段你只需记两件事：**用** skills 仓（下册装好），**有事**去应用仓/skills 仓提 Issue（第 3 册讲路径）。其余四仓，知道分工即可。

<img src="./images/01_six_repos.png" alt="CANNBot 六仓生态示意图" width="800px">

还有一点值得现在知道：每款 Skill 都按 [Agent Skills 开放标准](https://agentskills.io) 封装——**同一套知识可装进 8 类以上宿主**（OpenCode、Claude Code、Trae、Cursor…）。你用哪个工具，CANNBot 都跟得上。

</div>

<div align="left">

## 1.5 课后练习

本册介绍了编程 Agent 与 CANNBot 的基本概念，请根据学习内容完成以下题目自测。

1. （单选题）编程 Agent 与聊天 AI 最本质的区别是？  
    A. 编程 Agent 用的大模型参数量更大  
    B. 编程 Agent 能执行「写代码→运行→看结果→修正」的完整闭环，聊天 AI 到「给出回答」就停止  
    C. 编程 Agent 只能回答编程问题  
    D. 聊天 AI 不能生成代码

2. （多选题）关于 Skill（技能），下列说法正确的有哪些？  
    A. Skill 是写给 Agent 看的领域知识手册，干活前按需加载  
    B. Skill 只能由官方团队编写，社区开发者不能贡献  
    C. CANNBot 的 Skill 遵循 Agent Skills 开放标准，可装进多种宿主  
    D. 把所有 Skill 一次性全部塞给模型是最佳实践

3. （单选题）「Plugin 像菜谱，Agent 像厨师，Skill 像手册」——新增一个开发场景主要需要新增什么？  
    A. 新增一批 Agent（厨师）  
    B. 新增一个 Plugin（菜谱），复用现有 Agent 与 Skill  
    C. 重写所有 Skill  
    D. 更换底层大模型

4. （单选题）Agent 生成的代码不能保证完全正确。面对这一局限，CANNBot 的应对方式是？  
    A. 限制能力——只允许生成简单代码，复杂算子必须由人编写  
    B. 把「独立验收」设计进工作流——测试先行、真机验证、验收证据可复现  
    C. 在输出中提示用户自行承担全部风险  
    D. 依赖大模型能力提升，待未来版本自然解决

**执行以下代码获取答案。**

</div>

In [ ]:
!cat ./answer/01_answer.txt

<div align="left">

## 本册小结

- **闭环是分界线**：聊天 AI 出主意，编程 Agent 干活（读写文件、执行命令、自我验证）；
- **三个关键词**：Agent（大脑+手脚+自主决策）、Skill（按需加载的手册，社区共建）、多 Agent 协作（执行与验收分离）；
- **CANNBot**：面向 CANN 开发的智能体系，三层架构（Plugin 编排 / Agent 执行 / Skill 供能），Flash 轻量版八阶段工作流；
- **六仓生态**：你用 skills 仓、有事提 Issue，其余各司其职；Skill 开放标准让你在任何宿主都能用。

概念地基打好了。下一册：[第一个算子——从安装到验收的完整体验](./02_flash_workflow.ipynb)，亲手让 CANNBot 交付你的第一个算子。

</div>